info txt for running 

In [ ]:
import os

wheels_path = "/kaggle/input/notebooks/ttahara/birdclef-2026-download-wheels/wheels"
if os.path.exists(wheels_path):
    print("Installing libraries offline...")
    !pip install {wheels_path}/*.whl --no-index --find-links={wheels_path}
    print("Done!")
else:
    print(f"Error: Folder {wheels_path} not found. Check the name in Data.")

import json
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T

import onnxruntime as ort

warnings.filterwarnings("ignore")

class CFG:
    BASE_DIR     = "/kaggle/input/competitions/birdclef-2026"
    TEST_DIR     = f"{BASE_DIR}/test_soundscapes"
    SAMPLE_SUB   = f"{BASE_DIR}/sample_submission.csv"

    DATASET_DIR  = "/kaggle/input/datasets/studentedvard/birdclef-model-v1-zvuk2" 
    
    ONNX_PATH    = f"{DATASET_DIR}/bird_model.onnx"
    META_PATH    = f"{DATASET_DIR}/target_columns.json"

    TARGET_SR    = 32_000
    SEGMENT_SEC  = 5.0
    N_MELS       = 128
    N_FFT        = 1024
    HOP_LENGTH   = 320
    FMIN         = 20.0
    FMAX         = 16_000.0

cfg = CFG()

if not Path(cfg.META_PATH).exists():
    raise FileNotFoundError(f"Metadata file not found: {cfg.META_PATH}. Check DATASET_DIR path.")

with open(cfg.META_PATH) as f:
    INF_TARGET_COLUMNS = json.load(f)
print(f" Loaded {len(INF_TARGET_COLUMNS)} bird classes from {cfg.META_PATH}")

def build_mel_transforms(cfg: CFG):
    mel = T.MelSpectrogram(
        sample_rate = cfg.TARGET_SR,
        n_fft       = cfg.N_FFT,
        hop_length  = cfg.HOP_LENGTH,
        n_mels      = cfg.N_MELS,
        f_min       = cfg.FMIN,
        f_max       = cfg.FMAX,
    )
    db = T.AmplitudeToDB(stype="power", top_db=80)
    return mel, db

def audio_to_spec(wav_segment: torch.Tensor, mel_t, db_t) -> np.ndarray:
    spec = db_t(mel_t(wav_segment))
    spec = (spec - spec.mean()) / (spec.std() + 1e-6)
    return spec.unsqueeze(0).numpy()

def predict_file(path: str, session, mel_t, db_t, cfg: CFG) -> list:
    fname = Path(path).stem
    try:
        wav, sr = torchaudio.load(path)
    except Exception as e:
        print(f"[ERROR] Error loading {path}: {e}")
        return []

    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)

    if sr != cfg.TARGET_SR:
        wav = T.Resample(orig_freq=sr, new_freq=cfg.TARGET_SR)(wav)

    seg_len    = int(cfg.SEGMENT_SEC * cfg.TARGET_SR)
    input_name = session.get_inputs()[0].name
    results    = []

    num_frames = wav.shape[1]
    n_windows = int(np.ceil(num_frames / seg_len))
    
    if n_windows == 0:
        return []

    for i in range(n_windows):
        start = i * seg_len
        end   = start + seg_len

        segment = wav[:, start:end]

        if segment.shape[1] < seg_len:
            segment = F.pad(segment, (0, seg_len - segment.shape[1]))

        spec  = audio_to_spec(segment, mel_t, db_t)
        probs = session.run(None, {input_name: spec})[0]
        probs = probs[0]

        row_id = f"{fname}_{(i + 1) * 5}"
        results.append((row_id, probs))

    return results

if not Path(cfg.ONNX_PATH).exists():
    raise FileNotFoundError(f"ONNX model not found at path: {cfg.ONNX_PATH}. Check dataset name.")

session = ort.InferenceSession(
    cfg.ONNX_PATH,
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
    if torch.cuda.is_available()
    else ["CPUExecutionProvider"],
)
print(f" ONNX session active. Providers: {session.get_providers()}")

mel_t, db_t = build_mel_transforms(cfg)
test_files  = sorted(glob.glob(f"{cfg.TEST_DIR}/*.ogg"))
print(f" Found test files: {len(test_files)}")

sample_sub = pd.read_csv(cfg.SAMPLE_SUB)

if test_files:
    all_rows = []
    for fp in tqdm(test_files, desc="Inference"):
        rows = predict_file(fp, session, mel_t, db_t, cfg)
        all_rows.extend(rows)

    submission_df = pd.DataFrame(
        [{"row_id": r, **dict(zip(INF_TARGET_COLUMNS, p))} for r, p in all_rows]
    )

    for col in sample_sub.columns:
        if col not in submission_df.columns:
            submission_df[col] = 0.0
            
    submission_df = submission_df[sample_sub.columns]

    submission_df.to_csv("submission.csv", index=False)
    print(f"success. submission.csv saved. Rows: {len(submission_df):,}")

else:
    print(" Test data missing (development mode). Creating dummy submission.")
    for col in INF_TARGET_COLUMNS:
        if col in sample_sub.columns:
            sample_sub[col] = 0.5
            
    sample_sub.to_csv("submission.csv", index=False)
    print(f"success. Dummy submission.csv saved.")